In [1]:
1+1

2

### Data Ingestion

In [2]:
import os
%pwd

'c:\\Users\\HP\\Text-Summarizer\\research'

In [3]:
os.chdir("../")
%pwd

'c:\\Users\\HP\\Text-Summarizer'

### 1. Config Entity

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    ingested_train_dir: Path
    ingested_test_dir: Path
    dataset_name: str

    

### 2. Configuration Manager

In [14]:
from src.textsummarizer.utils.common import read_yaml, create_directories


CONFIG_FILE_PATH = Path("config/config.yaml")
PARAMS_FILE_PATH = Path("params.yaml")

class ConfigurationManager:
    def __init__(self):
        self.config = read_yaml(CONFIG_FILE_PATH)
        self.params = read_yaml(PARAMS_FILE_PATH)

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.artifacts.data_ingestion
        os.makedirs(config.root_dir, exist_ok=True)
        data_ingestion_config = DataIngestionConfig(
            root_dir=Path(config.root_dir),
            ingested_train_dir=Path(config.ingested_train_dir),
            ingested_test_dir=Path(config.ingested_test_dir),
            dataset_name=config.dataset_name
        )
        return data_ingestion_config

### 3. Data Ingestion Component

In [17]:
from datasets import load_dataset
from src.textsummarizer.logging import logger

class DataIngestion:
    def __init__(self, config:DataIngestionConfig):
        self.config = config

    def download_and_save(self):
        logger.info(f"Downloading dataset: {self.config.dataset_name}")
        dataset = load_dataset(self.config.dataset_name)

        os.makedirs(self.config.ingested_train_dir, exist_ok=True)
        os.makedirs(self.config.ingested_test_dir, exist_ok=True)

        dataset['train'].to_csv(os.path.join(self.config.ingested_train_dir, "train.csv"), index=False)
        dataset['test'].to_csv(os.path.join(self.config.ingested_test_dir, "test.csv"), index=False)

        logger.info(f'Train samples: {len(dataset["train"])}')
        logger.info(f'Test samples: {len(dataset["test"])}')
        logger.info(f'Data ingestion complete.')
        return dataset

### 4. Run the Pipeline

In [19]:
try:
    config_manager = ConfigurationManager()
    data_ingestion_config = config_manager.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    dataset = data_ingestion.download_and_save()
except Exception as e:
    raise e

[2026-05-22 09:54:57,560 - INFO - YAML file: config\config.yaml loaded successfully]
[2026-05-22 09:54:57,562 - INFO - YAML file: params.yaml loaded successfully]
[2026-05-22 09:54:57,564 - INFO - Downloading dataset: knkarthick/samsum]


c:\Users\HP\Text-Summarizer\venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\datasets--knkarthick--samsum. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Creating CSV from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 77.05ba/s]

[2026-05-22 09:55:08,608 - INFO - Train samples: 14731]
[2026-05-22 09:55:08,609 - INFO - Test samples: 819]
[2026-05-22 09:55:08,609 - INFO - Data ingestion complete.]


### Sanity Check

In [20]:
print('Dataset features:', dataset['train'].features)
print('\nSample record:')
print(dataset['train'][0])

Dataset features: {'id': Value('string'), 'dialogue': Value('string'), 'summary': Value('string')}

Sample record:
{'id': '13818513', 'dialogue': "Amanda: I baked  cookies. Do you want some?\nJerry: Sure!\nAmanda: I'll bring you tomorrow :-)", 'summary': 'Amanda baked cookies and will bring Jerry some tomorrow.'}
